# 05 · Flat 39-target baseline

One Swin-V2-B with 39 independent sigmoid outputs, trained on **all** images
(healthy images have all-zero targets). Backbone, resolution, preprocessing,
augmentation, loss, optimiser, schedule, early stopping, splits and metrics are
identical to the cascade; only the architecture differs.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # or: pip install -e ..

import numpy as np, pandas as pd, torch
from fundus import config, labels
from fundus.data import (load_split, unhealthy_only, FundusDataset, make_loader,
                         train_transform, eval_transform, pos_weight)
from fundus.engine import set_seed, device, fit, predict

P = config.Paths()          # reads $FUNDUS_DATA_ROOT and $FUNDUS_RUN_ROOT
set_seed(config.SEED)
DEV = device()
print("device:", DEV, "| data:", P.data_root.resolve())

In [ ]:
from fundus.models import build_classifier, load_weights
from fundus.losses import make_criterion
from fundus.engine import evaluate_multilabel
from fundus.metrics import tune_thresholds, score_all, per_class_table

cfg, TG = config.FLAT, labels.TARGETS
tr, va, te = (load_split(P.csv(s)) for s in ("train", "val", "test"))
train_ds = FundusDataset(tr, P.images("train"), train_transform(cfg.img_size), label_cols=TG)
val_ds   = FundusDataset(va, P.images("val"),   eval_transform(cfg.img_size),  label_cols=TG)
test_ds  = FundusDataset(te, P.images("test"),  eval_transform(cfg.img_size),  label_cols=TG)
train_dl = make_loader(train_ds, cfg.batch_size, True,  cfg.num_workers)
val_dl   = make_loader(val_ds,   cfg.batch_size, False, cfg.num_workers)
test_dl  = make_loader(test_ds,  cfg.batch_size, False, cfg.num_workers)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)} | "
      f"all-zero train targets {(train_ds.labels.sum(1) == 0).sum()}")

In [ ]:
OUT = P.run("flat")
model = build_classifier(len(TG), cfg)
criterion = make_criterion("bce", pos_weight(train_ds.labels, cfg.pos_weight_clip))
hist = fit(model, train_dl, val_dl, criterion, evaluate_multilabel, OUT, cfg,
           monitors=("macro_auc", "macro_ap"))

## Thresholds on validation, single evaluation on test

In [ ]:
model = load_weights(model, OUT / "best_macro_ap.pt", DEV)
Yv, Pv, _ = predict(model, val_dl, DEV)
Yt, Pt, St = predict(model, test_dl, DEV)
TH = tune_thresholds(Yv, Pv, min_pos=10)
json.dump({"thresholds": dict(zip(TG, TH.tolist()))}, open(OUT / "thresholds.json", "w"), indent=2)

print(pd.Series(score_all(Yt, Pt, TH, "flat")).to_string())
per_class_table(Yt, Pt, TH, TG).to_csv(OUT / "flat_per_disease.csv")
np.save(OUT / "Pv.npy", Pv); np.save(OUT / "Pt.npy", Pt); np.save(OUT / "Yv.npy", Yv); np.save(OUT / "Yt.npy", Yt)